# 60 - Scaling Evaluation: Recall@k / NDCG@k and Latency Across Corpus Tiers

Answers the first of the two questions raised in the scaling-corpus section: does retrieval quality and latency hold up as the searchable pool grows from 100K to ~397K companies? Covers BM25 (from notebook 59) plus the three embedding models already inside the trained fusion ranker (MiniLM, GTE-large, Linq-Embed-Mistral), evaluated against the combined gold standard restricted to the 19 deep-coverage queries (5-query pilot + 14 headline queries), the only queries with enough judged depth to report Recall@k/NDCG@k without excessive variance. Every gold-relevant candidate for these 19 queries is present in all four tiers by construction (Section~44's nested tier design), so Recall@k is directly comparable across tiers -- any change is attributable to added distractor companies, not to relevant companies dropping out of the pool.

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

OUTPUT_DIR = Path("result/60_scaling_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIERS = [("100k", "in_100k"), ("200k", "in_200k"), ("300k", "in_300k"), ("400k", "in_400k")]
DEEP_QUERY_IDS = [1, 2, 3, 4, 5, 11, 12, 15, 34, 14, 27, 66, 72, 99, 101, 56, 82, 91, 92]
K_VALUES = [10, 50, 100, 300, 500, 1000]
TOP_K = 1000
RELEVANT_THRESHOLD = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Setup] Device: {DEVICE}")

train_queries = json.load(open("result/08_llm_relevance_judge/train_queries.json"))
held_out_queries = json.load(open("result/08_llm_relevance_judge/held_out_queries.json"))
query_lookup = {item["query_id"]: item["query"] for item in train_queries + held_out_queries}
deep_queries = [(qid, query_lookup[qid]) for qid in DEEP_QUERY_IDS]

base_gold = pd.read_json("result/40_active_learning_labeling_queue/expanded_gold_labels.json")
round_gold = pd.read_json("result/42_headline_query_deepening/round_gold_labels.json")
gold = pd.concat([base_gold[["query_id", "domain", "gold_label"]], round_gold[["query_id", "domain", "gold_label"]]], ignore_index=True).drop_duplicates(subset=["query_id", "domain"])
gold = gold[gold["query_id"].isin(DEEP_QUERY_IDS)]
print(f"[Load] Gold standard for the 19 deep queries: {len(gold):,} candidates")

combined = pd.read_parquet("result/44_build_scaled_corpus/combined_pool.parquet")
print(f"[Load] Combined pool: {len(combined):,} companies")

In [ ]:
from sentence_transformers import SentenceTransformer

os.environ["HF_HUB_OFFLINE"] = "1"
query_texts = [qtext for _, qtext in deep_queries]

print("[Encode] Loading MiniLM for query encoding...")
try:
    minilm_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE, local_files_only=True)
except Exception:
    minilm_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
minilm_query_embs = minilm_model.encode(query_texts, normalize_embeddings=False, convert_to_numpy=True).astype("float32")
print(f"[Encode] MiniLM query embeddings: {minilm_query_embs.shape}")

print("[Encode] Loading GTE-large for query encoding...")
try:
    gte_model = SentenceTransformer("thenlper/gte-large", device=DEVICE, local_files_only=True)
except Exception:
    gte_model = SentenceTransformer("thenlper/gte-large", device=DEVICE)
gte_query_embs = gte_model.encode(query_texts, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
print(f"[Encode] GTE-large query embeddings: {gte_query_embs.shape}")

del minilm_model, gte_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

In [ ]:
from transformers import AutoTokenizer, AutoModel

TASK_INSTRUCTION = "Given a search query describing a type of company, retrieve relevant company profiles"
query_prefix = f"Instruct: {TASK_INSTRUCTION}\nQuery: "
MAX_LENGTH = 512
REPO = "Linq-AI-Research/Linq-Embed-Mistral"

print("[Encode] Loading Linq-Embed-Mistral for query encoding...")
try:
    linq_tokenizer = AutoTokenizer.from_pretrained(REPO, local_files_only=True)
    linq_model = AutoModel.from_pretrained(REPO, torch_dtype=torch.float16, device_map=DEVICE, local_files_only=True)
    print("[Encode] Loaded from local cache -- skipped Hugging Face Hub network calls")
except Exception:
    print("[Encode] Not fully cached locally yet -- retrying with network access")
    linq_tokenizer = AutoTokenizer.from_pretrained(REPO)
    linq_model = AutoModel.from_pretrained(REPO, torch_dtype=torch.float16, device_map=DEVICE)
linq_model.eval()


def last_token_pool(last_hidden_states, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


@torch.no_grad()
def encode_linq(texts):
    batch_dict = linq_tokenizer(texts, max_length=MAX_LENGTH, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
    outputs = linq_model(**batch_dict)
    embs = last_token_pool(outputs.last_hidden_state, batch_dict["attention_mask"])
    embs = F.normalize(embs, p=2, dim=1)
    return embs.cpu().float().numpy()


query_texts_with_instruction = [query_prefix + qtext for qtext in query_texts]
linq_query_embs = encode_linq(query_texts_with_instruction).astype("float32")
print(f"[Encode] Linq-Embed-Mistral query embeddings: {linq_query_embs.shape}")

del linq_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

In [ ]:
EMBEDDING_METHODS = {
    # name -> (company_embeddings_path, query_embeddings, metric)  metric: "l2" (lower better) or "cosine" (higher better, dot product since normalized)
    "minilm": ("result/45_encode_minilm_scaled/company_embeddings.npy", minilm_query_embs, "l2"),
    "gte": ("result/46_encode_gte_scaled/company_embeddings.npy", gte_query_embs, "cosine"),
    "linq": ("result/47_encode_linq_mistral_scaled/company_embeddings.npy", linq_query_embs, "cosine"),
}

all_embedding_rows = []
timing_rows = []

for method_name, (emb_path, query_embs, metric) in EMBEDDING_METHODS.items():
    out_path = OUTPUT_DIR / f"{method_name}_results_all_tiers.json"
    if out_path.exists():
        print(f"[{method_name}] Already done -- loading from disk")
        method_results = pd.read_json(out_path)
        all_embedding_rows.append(method_results)
        timing_path = OUTPUT_DIR / f"{method_name}_timing.json"
        if timing_path.exists():
            timing_rows.append(pd.read_json(timing_path))
        continue

    print(f"[{method_name}] Loading company embeddings from {emb_path}...")
    full_embs = np.load(emb_path, mmap_mode="r")

    method_rows = []
    method_timing = []
    for tier_name, flag_col in TIERS:
        mask = combined[flag_col].values
        tier_domains = combined.loc[mask, "domain"].reset_index(drop=True)
        tier_embs = np.asarray(full_embs[mask]).astype("float32")
        print(f"[{method_name}] Tier {tier_name}: {tier_embs.shape[0]:,} companies")

        for (qid, qtext), q_emb in zip(deep_queries, query_embs):
            t0 = time.perf_counter()
            if metric == "l2":
                dists = np.sum((tier_embs - q_emb) ** 2, axis=1)
                top_idx = np.argsort(dists)[:TOP_K]
                scores = -dists[top_idx]  # keep "higher is better" convention downstream
            else:
                sims = tier_embs @ q_emb
                top_idx = np.argsort(sims)[::-1][:TOP_K]
                scores = sims[top_idx]
            query_ms = (time.perf_counter() - t0) * 1000

            for rank, (idx, score) in enumerate(zip(top_idx, scores)):
                method_rows.append({
                    "method": method_name, "tier": tier_name, "query_id": qid,
                    "rank": rank + 1, "domain": tier_domains.iloc[idx], "score": float(score),
                })
            method_timing.append({
                "method": method_name, "tier": tier_name, "query_id": qid,
                "n_companies": tier_embs.shape[0], "query_latency_ms": query_ms,
            })

    method_results = pd.DataFrame(method_rows)
    method_results.to_json(out_path, orient="records", indent=2)
    pd.DataFrame(method_timing).to_json(OUTPUT_DIR / f"{method_name}_timing.json", orient="records", indent=2)
    print(f"[{method_name}] Saved -> {out_path}")
    all_embedding_rows.append(method_results)
    timing_rows.append(pd.DataFrame(method_timing))
    del full_embs

print("[Done] All embedding methods processed.")

In [ ]:
bm25_all = pd.read_json("result/59_bm25_tiered_index/bm25_results_all_tiers.json")
bm25_all["method"] = "bm25"
bm25_all = bm25_all[["method", "tier", "query_id", "rank", "domain", "bm25_score"]].rename(columns={"bm25_score": "score"})

bm25_timing_files = sorted(Path("result/59_bm25_tiered_index").glob("bm25_timing_*.json"))
bm25_timing = pd.concat([pd.read_json(f) for f in bm25_timing_files], ignore_index=True)
bm25_timing["method"] = "bm25"

all_results = pd.concat([bm25_all] + all_embedding_rows, ignore_index=True)
all_timing = pd.concat([bm25_timing] + timing_rows, ignore_index=True)
all_results.to_json(OUTPUT_DIR / "all_methods_all_tiers.json", orient="records", indent=2)
all_timing.to_csv(OUTPUT_DIR / "all_methods_timing.csv", index=False)
print(f"[Combined] {len(all_results):,} retrieval rows, {len(all_timing):,} timing rows")
print(f"[Combined] Methods: {sorted(all_results['method'].unique())}")
print(f"[Combined] Tiers: {sorted(all_results['tier'].unique())}")

In [ ]:
relevant_by_query = {
    qid: set(gold[(gold["query_id"] == qid) & (gold["gold_label"] >= RELEVANT_THRESHOLD)]["domain"])
    for qid in DEEP_QUERY_IDS
}


def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0


def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else None


def dcg_at_k(retrieved, relevant, k):
    return sum(1 / np.log2(i + 2) for i, d in enumerate(retrieved[:k]) if d in relevant)


def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else None


eval_rows = []
for method_name in all_results["method"].unique():
    for tier_name, _ in TIERS:
        sub = all_results[(all_results["method"] == method_name) & (all_results["tier"] == tier_name)]
        for qid in DEEP_QUERY_IDS:
            relevant = relevant_by_query[qid]
            retrieved = sub[sub["query_id"] == qid].sort_values("rank")["domain"].tolist()
            for k in K_VALUES:
                eval_rows.append({
                    "method": method_name, "tier": tier_name, "query_id": qid, "k": k,
                    "precision": precision_at_k(retrieved, relevant, k),
                    "recall": recall_at_k(retrieved, relevant, k),
                    "ndcg": ndcg_at_k(retrieved, relevant, k),
                })

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(OUTPUT_DIR / "scaling_eval_per_query.csv", index=False)

summary = eval_df.groupby(["method", "tier", "k"])[["precision", "recall", "ndcg"]].mean().reset_index()
summary["tier"] = pd.Categorical(summary["tier"], categories=["100k", "200k", "300k", "400k"], ordered=True)
summary = summary.sort_values(["method", "tier", "k"])
summary.to_csv(OUTPUT_DIR / "scaling_eval_summary.csv", index=False)
print(summary.to_string(index=False))

In [ ]:
latency_summary = all_timing.groupby(["method", "tier"]).agg(n_companies=("n_companies", "first"), avg_query_latency_ms=("query_latency_ms", "mean")).reset_index()
latency_summary["tier"] = pd.Categorical(latency_summary["tier"], categories=["100k", "200k", "300k", "400k"], ordered=True)
latency_summary = latency_summary.sort_values(["method", "tier"])
latency_summary.to_csv(OUTPUT_DIR / "latency_summary.csv", index=False)
print(latency_summary.to_string(index=False))